In [10]:
from pathlib import Path
import warnings
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torchvision import models, transforms

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device} | torch: {torch.__version__}")

device: cpu | torch: 2.14.1+cpu


In [11]:
BASE_DIR = Path("..").resolve()
sys.path.insert(0, str(BASE_DIR / "src" / "cv"))

PROCESSED_DIR = BASE_DIR / "data" / "cv" / "processed"
SPLIT_DIRS = {
    "train": PROCESSED_DIR / "train_v2",
    "val": PROCESSED_DIR / "val_v2",
    "test": PROCESSED_DIR / "test_v2",
}
FRAMES_DIR = PROCESSED_DIR / "frames_effnet30"  
LABELS_CSV = BASE_DIR / "data" / "cv" / "raw" / "solution.csv"
CACHE_DIR = BASE_DIR / "outputs" / "feature_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
N_FRAMES = 30

for p in [*SPLIT_DIRS.values(), FRAMES_DIR, LABELS_CSV]:
    assert p.exists(), f"Path not found: {p}"
print("BASE_DIR:", BASE_DIR)

BASE_DIR: C:\Users\98936\Desktop\incident-sentiment-analyzer


In [3]:
ldf = pd.read_csv(LABELS_CSV, dtype={"id": str})
ldf["id"] = ldf["id"].str.zfill(5)
label_map = dict(zip(ldf["id"], ldf["target"].astype(int)))
group_map = dict(zip(ldf["id"], ldf["group"]))

rows = []
for split, d in SPLIT_DIRS.items():
    for cls in ["negative", "positive"]:
        for f in sorted((d / cls).glob("*.mp4")):
            vid = f.stem.zfill(5)
            rows.append(
                dict(
                    id=vid,
                    split=split,
                    folder=cls,
                    label=label_map.get(vid, -1),
                    group=group_map.get(vid, -1),
                )
            )
meta = pd.DataFrame(rows)

assert meta["id"].is_unique, "a video id appears in more than one split!"
assert (meta["label"] >= 0).all(), "some videos are missing from solution.csv"
assert (meta["group"] >= 0).all(), "some videos are missing from solution.csv"

print(meta.groupby(["split", "label"]).size().unstack())
print(meta.groupby(["split", "group"]).size().unstack())
print("folder vs CSV label (for the record):")
print(pd.crosstab(meta["folder"], meta["label"]))

missing = [
    v for v in meta["id"] if len(list((FRAMES_DIR / v).glob("*.jpg"))) != N_FRAMES
]
print("videos without exactly", N_FRAMES, "frames:", len(missing))
assert not missing, missing[:5]

label   0   1
split        
test    7  16
train  35  70
val     8  14
group    0
split     
test    23
train  105
val     22
folder vs CSV label (for the record):
label      0    1
folder           
negative  50    0
positive   0  100
videos without exactly 30 frames: 0


In [4]:
mis = meta[meta["folder"] != meta["label"].map({0: "negative", 1: "positive"})]
print("folder vs CSV mismatches:", len(mis))
assert len(mis) == 0, mis[["id", "split", "folder", "label"]]

folder vs CSV mismatches: 0


In [5]:
EXPECTED_D = {
    "efficientnet_b0": 1280,
    "efficientnet_b3": 1536,
    "convnext_tiny": 768,
    "resnet50": 2048,
}

def make_extractor(name):
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier = nn.Identity()
    elif name == "efficientnet_b3":
        m = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
        m.classifier = nn.Identity()
    elif name == "convnext_tiny":
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        # classifier = [LayerNorm2d, Flatten, Linear] -> drop only the Linear
        m.classifier[2] = nn.Identity()
    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
    else:
        raise ValueError(f"Unknown backbone: {name}")
    return m.eval().to(device)


prep = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def load_frames(vid):
    paths = sorted((FRAMES_DIR / vid).glob("*.jpg"))
    assert len(paths) == N_FRAMES, f"{vid}: found {len(paths)} frames"
    return torch.stack([prep(Image.open(p).convert("RGB")) for p in paths])


@torch.no_grad()
def get_features(name):
    """Extract [n_videos, N_FRAMES, D] following meta. Cached to disk."""
    cache = CACHE_DIR / f"{name}_{N_FRAMES}f.npy"
    want = (len(meta), N_FRAMES, EXPECTED_D[name])
    if cache.exists():
        F = np.load(cache)
        if F.shape == want:
            return F
        print(f"  stale cache for {name}: {F.shape} != {want}, re-extracting")

    model = make_extractor(name)
    out = []
    for i, vid in enumerate(meta["id"]):
        out.append(model(load_frames(vid).to(device)).cpu().numpy())
        if (i + 1) % 5 == 0:
            print(f"  {name}: {i + 1}/{len(meta)}")

    F = np.stack(out).astype(np.float32)
    assert F.shape == want, f"{name}: got {F.shape}, expected {want}"
    np.save(cache, F)
    return F


BACKBONES = ["efficientnet_b0", "resnet50", "convnext_tiny", "efficientnet_b3"]
features = {}
for b in BACKBONES:
    features[b] = get_features(b)
    print(b, features[b].shape)

efficientnet_b0 (150, 30, 1280)
resnet50 (150, 30, 2048)
convnext_tiny (150, 30, 768)
efficientnet_b3 (150, 30, 1536)


In [6]:
def mean_group_ap(y, s, g):
    """Mean group AP. Single-class groups are skipped."""
    aps = []
    for gv in np.unique(g):
        m = g == gv
        if len(np.unique(y[m])) < 2:
            continue
        aps.append(average_precision_score(y[m], s[m]))
    return float(np.mean(aps))


def aggregate(F, kind):
    """F: [N, T, D] -> [N, features]"""
    mean, mx, sd = F.mean(1), F.max(1), F.std(1)
    return {
        "mean": mean,
        "mean+max": np.hstack([mean, mx]),
        "mean+max+std": np.hstack([mean, mx, sd]),
    }[kind]


def make_clf(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))


def top_k_mean(scores, k=5):
    return np.sort(scores, axis=1)[:, -k:].mean(1)


def cv_eval(F, y, g, kind, C, n_splits=5, n_repeats=3, seed=0):
    """Repeated K-Fold cross-validation on videos.

    Returns: (mean, std) across repeats.
    """
    strat = y * 10 + g
    X = None if kind == "frame_top5" else aggregate(F, kind)
    scores = []
    for r in range(n_repeats):
        oof = np.zeros(len(y))
        skf = StratifiedKFold(n_splits, shuffle=True, random_state=seed + r)
        for tr, te in skf.split(np.zeros(len(y)), strat):
            clf = make_clf(C)
            if kind == "frame_top5":
                T, D = F.shape[1], F.shape[2]
                clf.fit(F[tr].reshape(-1, D), np.repeat(y[tr], T))
                oof[te] = top_k_mean(
                    clf.decision_function(F[te].reshape(-1, D)).reshape(len(te), T)
                )
            else:
                clf.fit(X[tr], y[tr])
                oof[te] = clf.decision_function(X[te])
        scores.append(mean_group_ap(y, oof, g))
    return float(np.mean(scores)), float(np.std(scores))


# Pool = train + val (test remains untouched)
pool = (meta["split"] != "test").to_numpy()
y_pool = meta.loc[pool, "label"].to_numpy()
g_pool = meta.loc[pool, "group"].to_numpy()
print("pool videos:", pool.sum(), "| positives:", y_pool.sum())
print(
    "min videos per (label, group):",
    pd.Series(y_pool * 10 + g_pool).value_counts().min(),
)

pool videos: 127 | positives: 84
min videos per (label, group): 43


In [7]:
C_GRID = [0.001, 0.01, 0.1]
POOLINGS = ["frame_top5", "mean", "mean+max", "mean+max+std"]

res_rows = []
for b in BACKBONES:
    F = features[b][pool]
    for kind in POOLINGS:
        for C in C_GRID:
            m, s = cv_eval(F, y_pool, g_pool, kind, C)
            res_rows.append(dict(backbone=b, pooling=kind, C=C, ap=m, std=s))
            print(f"{b:16s} {kind:13s} C={C:<6} AP={m:.3f} ± {s:.3f}", flush=True)

res = pd.DataFrame(res_rows).sort_values("ap", ascending=False).reset_index(drop=True)

print("\nBest C per (backbone, pooling):")
best_c = res.loc[res.groupby(["backbone", "pooling"])["ap"].idxmax()]
print(best_c.pivot(index="backbone", columns="pooling", values="ap").round(3))
print("\nTop 15:")
print(res.head(15).round(3).to_string())

efficientnet_b0  frame_top5    C=0.001  AP=0.728 ± 0.016
efficientnet_b0  frame_top5    C=0.01   AP=0.730 ± 0.014
efficientnet_b0  frame_top5    C=0.1    AP=0.735 ± 0.018
efficientnet_b0  mean          C=0.001  AP=0.686 ± 0.028
efficientnet_b0  mean          C=0.01   AP=0.702 ± 0.021
efficientnet_b0  mean          C=0.1    AP=0.709 ± 0.028
efficientnet_b0  mean+max      C=0.001  AP=0.682 ± 0.028
efficientnet_b0  mean+max      C=0.01   AP=0.687 ± 0.031
efficientnet_b0  mean+max      C=0.1    AP=0.691 ± 0.030
efficientnet_b0  mean+max+std  C=0.001  AP=0.675 ± 0.025
efficientnet_b0  mean+max+std  C=0.01   AP=0.684 ± 0.027
efficientnet_b0  mean+max+std  C=0.1    AP=0.681 ± 0.027
resnet50         frame_top5    C=0.001  AP=0.765 ± 0.007
resnet50         frame_top5    C=0.01   AP=0.764 ± 0.011
resnet50         frame_top5    C=0.1    AP=0.762 ± 0.014
resnet50         mean          C=0.001  AP=0.755 ± 0.010
resnet50         mean          C=0.01   AP=0.776 ± 0.011
resnet50         mean          

In [8]:
OUT_DIR = BASE_DIR / "outputs" / "cv_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

POOL_ORDER = {p: i for i, p in enumerate(POOLINGS)}
BB_ORDER = {b: i for i, b in enumerate(BACKBONES)}

grid_sorted = (
    res.assign(
        _bb=res["backbone"].map(BB_ORDER),
        _pl=res["pooling"].map(POOL_ORDER),
    )
    .sort_values(["_bb", "_pl", "C"])
    .drop(columns=["_bb", "_pl"])
    .round({"ap": 4, "std": 4})
    .reset_index(drop=True)
)

best = (
    grid_sorted.loc[grid_sorted.groupby(["backbone", "pooling"])["ap"].idxmax()]
    .sort_values(["ap"], ascending=False)
    .reset_index(drop=True)
)

grid_sorted.to_csv(OUT_DIR / "07_grid_all.csv", index=False)
best.to_csv(OUT_DIR / "07_best_c.csv", index=False)

print("saved:")
print(OUT_DIR / "07_grid_all.csv")
print(OUT_DIR / "07_best_c.csv")

saved:
C:\Users\98936\Desktop\incident-sentiment-analyzer\outputs\cv_results\07_grid_all.csv
C:\Users\98936\Desktop\incident-sentiment-analyzer\outputs\cv_results\07_best_c.csv


In [13]:
import joblib
import numpy as np
import pandas as pd
from evaluate_submission import score

# ---------- 1) Selection (CV-only) ----------
BACKBONE, POOLING, C_FINAL = "efficientnet_b0", "frame_top5", 0.1
assert POOLING == "frame_top5", "this cell implements frame_top5 only"

assert len(res) == 48, f"res has {len(res)} rows, expected 48"
r = res[(res.backbone == BACKBONE) & (res.pooling == POOLING) & (res.C == C_FINAL)]
assert len(r) == 1, (BACKBONE, POOLING, C_FINAL)
sel = r.iloc[0]
print(
    f"CV {BACKBONE}/{POOLING}/C={C_FINAL}: "
    f"{sel['ap']:.4f} ± {sel['std']:.4f}  (optimistic)"
)

# ---------- 2) Leakage guard & data shape check ----------
test = (meta["split"] == "test").to_numpy()
assert test.sum() == 23, test.sum()
assert (meta.loc[pool, "split"] != "test").all(), "pool includes test rows"
assert len(y_pool) == pool.sum()
assert meta["id"].astype(str).str.len().eq(5).all(), "id must be 5-char string"

# ---------- 3) Fit on train+val (frame-level), score test once ----------
F = features[BACKBONE]
T, D = F.shape[1], F.shape[2]
clf = make_clf(C_FINAL).fit(F[pool].reshape(-1, D), np.repeat(y_pool, T))
s_test = top_k_mean(
    clf.decision_function(F[test].reshape(-1, D)).reshape(test.sum(), T)
)
y_test = meta.loc[test, "label"].to_numpy()
g_test = meta.loc[test, "group"].to_numpy()
ap_test = mean_group_ap(y_test, s_test, g_test)

# ---------- 4) Bootstrap confidence interval ----------
rng = np.random.default_rng(0)
boot, skipped = [], 0
for _ in range(2000):
    i = rng.integers(0, len(y_test), len(y_test))
    if len(np.unique(y_test[i])) < 2:
        skipped += 1
        continue
    boot.append(mean_group_ap(y_test[i], s_test[i], g_test[i]))
lo, hi = np.percentile(boot, [2.5, 97.5])
print(
    f"TEST mAP = {ap_test:.4f}  95% CI [{lo:.4f}, {hi:.4f}]  "
    f"(n={len(y_test)}, valid boots={len(boot)}, skipped={skipped})"
)

# ---------- 5) Submission & test files ----------
OUT = BASE_DIR / "outputs"
OUT.mkdir(exist_ok=True)
sub = pd.DataFrame(
    {
        "id": meta.loc[test, "id"].astype(str).to_numpy(),
        "score": s_test,
    }
)
sub_path = OUT / "submission.csv"
sub.to_csv(sub_path, index=False)

# ---------- 6) Official scorer on test IDs only ----------
sub_r = pd.read_csv(sub_path, dtype={"id": str})
sol_r = pd.read_csv(LABELS_CSV, dtype={"id": str})
sol_r = sol_r[sol_r["id"].isin(sub_r["id"])].reset_index(drop=True)
assert len(sol_r) == len(sub_r) == 23
sol_path = OUT / "solution_test_only.csv"
sol_r.to_csv(sol_path, index=False)
official = score(sol_r[["id", "target", "group"]], sub_r[["id", "score"]])
print(
    f"official score() = {official:.6f}  |  "
    f"diff vs local = {abs(official - ap_test):.2e}"
)
assert abs(official - ap_test) < 1e-6, "local mAP and official score() disagree"

# ---------- 7) Save model ----------
model_path = OUT / "final_model.joblib"
joblib.dump(
    dict(
        clf=clf,
        backbone=BACKBONE,
        pooling=POOLING,
        C=C_FINAL,
        n_frames=N_FRAMES,
        top_k=5,
        cv_ap=float(sel["ap"]),
        cv_std=float(sel["std"]),
        test_ap=float(ap_test),
    ),
    model_path,
)
print("saved:", sub_path, sol_path, model_path)

CV efficientnet_b0/frame_top5/C=0.1: 0.7349 ± 0.0177  (optimistic)
TEST mAP = 0.8119  95% CI [0.5988, 0.9671]  (n=23, valid boots=2000, skipped=0)
official score() = 0.811933  |  diff vs local = 0.00e+00
saved: C:\Users\98936\Desktop\incident-sentiment-analyzer\outputs\submission.csv C:\Users\98936\Desktop\incident-sentiment-analyzer\outputs\solution_test_only.csv C:\Users\98936\Desktop\incident-sentiment-analyzer\outputs\final_model.joblib
